<a href="https://colab.research.google.com/github/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots/blob/feature%2Fdata-acquisition/Making_the_Most_of_your_Colab_Subscription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS1090B — Hallucination in Legal RAG Chatbots — End-to-End Pipeline

Runs the data pipeline described in the README on Google Colab Pro A100.
Stages: bootstrap → env check → CourtListener ingest → data audit → dataset probe → LePaRD ingest → LePaRD↔CL compat audit.

In [1]:
# Cell 2: Bootstrap repo + uv venv (pipeline cells will subprocess via .venv/bin/python)
import os, subprocess, pathlib

REPO_URL = "https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git"
REPO_DIR = "/content/cs1090b_HallucinationLegalRAGChatbots"

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")

if not pathlib.Path(REPO_DIR).exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

sh("uv python install 3.11.9")
sh("uv python pin 3.11.9")
if not pathlib.Path(".venv").exists():
    sh("uv sync")

pathlib.Path(".env").write_text(
    "export PYTHONHASHSEED=0\n"
    "export CUBLAS_WORKSPACE_CONFIG=:4096:8\n"
    "export TOKENIZERS_PARALLELISM=false\n"
    "export RANDOM_SEED=0\n"
    "export TARGET_GPU_COUNT=1\n"
    "export TARGET_VRAM_GB_MIN=20.0\n"
)

# Verify .venv has correct pinned versions
r = subprocess.run(
    [".venv/bin/python", "-c",
     "import sys, numpy, torch, transformers; "
     "print(f'python {sys.version.split()[0]}'); "
     "print(f'numpy {numpy.__version__}'); "
     "print(f'torch {torch.__version__}'); "
     "print(f'transformers {transformers.__version__}')"],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr); raise SystemExit("venv verification failed")

$ git clone https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git /content/cs1090b_HallucinationLegalRAGChatbots
cwd: /content/cs1090b_HallucinationLegalRAGChatbots
$ uv python install 3.11.9
$ uv python pin 3.11.9
$ uv sync
python 3.11.9
numpy 1.26.4
torch 2.0.1+cu117
transformers 4.41.2



# Login with a web browser, at Terminal prompt
git config --global user.email "ltphongssvn@gmail.com" && git config --global user.name "ltphongssvn"


gh auth login --hostname github.com --git-protocol https --web

In [4]:

# Cell 3: Environment verification — runs notebooks/cs1090b_HallucinationLegalRAGChatbots.md Cell 1
import subprocess, os, sys, re, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

nb_src = pathlib.Path("notebooks/cs1090b_HallucinationLegalRAGChatbots.md").read_text()
blocks = re.findall(r"```\{code-cell\}[^\n]*\n(.*?)```", nb_src, flags=re.DOTALL)
assert blocks, "no code cells found in notebook md"
cell1_code = blocks[0].replace(
    "os.chdir(os.path.dirname(os.getcwd()))", "# chdir not needed — already at repo root"
)

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", cell1_code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 3 failed with exit code {proc.returncode}")

  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)
    TDD RED→GREEN: Environment Contract
  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 (allocated)
  ✓ PASS: GPU[0] NVIDIA A100-SXM4-80GB | cap (8, 0) | 85.1GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 201.9GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro

In [7]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Check free space on Drive
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB")

# Create target dir:
import pathlib
dst = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
dst.mkdir(parents=True, exist_ok=True)
print(dst, "ready")

Mounted at /content/drive
Drive free: 191.8 GB / total: 259.7 GB
/content/drive/MyDrive/cs1090b_cl_bulk ready


In [11]:
# Replace local dir with symlink to Drive
import pathlib
src = pathlib.Path("/content/cs1090b_HallucinationLegalRAGChatbots/data/raw/cl_bulk")
dst = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")

src.parent.mkdir(parents=True, exist_ok=True)

if src.is_symlink():
    print(f"already symlinked -> {src.resolve()}")
elif src.exists():
    src.rmdir()
    src.symlink_to(dst, target_is_directory=True)
    print(f"created {src} -> {dst}")
else:
    src.symlink_to(dst, target_is_directory=True)
    print(f"created {src} -> {dst}")

for p in sorted(dst.glob("*.csv.bz2")):
    print(f"  {p.name}  {p.stat().st_size/1e9:.2f} GB")

already symlinked -> /content/drive/MyDrive/cs1090b_cl_bulk
  courts-2026-03-31.csv.bz2  0.00 GB
  dockets-2026-03-31.csv.bz2  4.98 GB
  opinion-clusters-2026-03-31.csv.bz2  2.45 GB
  opinions-2026-03-31.csv.bz2  54.19 GB


In [14]:
# Cell 4: CourtListener bulk CSV download — Drive-persistent, skip if present
import os, sys, subprocess, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

DRIVE_BULK = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_bulk")
DRIVE_BULK.mkdir(parents=True, exist_ok=True)

local_bulk = pathlib.Path("data/raw/cl_bulk")
local_bulk.parent.mkdir(parents=True, exist_ok=True)
if local_bulk.exists() and not local_bulk.is_symlink():
    if local_bulk.is_dir() and not any(local_bulk.iterdir()):
        local_bulk.rmdir()
    else:
        raise SystemExit(f"{local_bulk} exists and is not empty/symlink")
if not local_bulk.exists():
    local_bulk.symlink_to(DRIVE_BULK, target_is_directory=True)
print(f"bulk_dir -> {local_bulk.resolve()}")

code = r'''
import logging
from src.config import PipelineConfig
from src.s3_discovery import discover_latest_bulk_files
from src.bulk_download import download_bulk_csvs

logging.basicConfig(level=logging.INFO, format="  %(message)s")
log = logging.getLogger("bulk")

cfg = PipelineConfig()
cfg.bulk_dir.mkdir(parents=True, exist_ok=True)

need = {"courts-", "dockets-", "opinion-clusters-", "opinions-"}
have = {p.name for p in cfg.bulk_dir.glob("*.csv.bz2")}
matched = {lbl for lbl in need if any(n.startswith(lbl) for n in have)}

if matched == need:
    log.info("All 4 bulk CSVs already present in %s - skipping" % cfg.bulk_dir)
    for p in sorted(cfg.bulk_dir.glob("*.csv.bz2")):
        log.info("  %s  %.2f GB" % (p.name, p.stat().st_size / 1e9))
else:
    log.info("Missing: %s" % sorted(need - matched))
    latest = discover_latest_bulk_files(config=cfg)
    for label, info in latest.items():
        log.info("  %-10s %s (%.1f MB)" % (label, info["key"], info["size_mb"]))
    paths = download_bulk_csvs(latest, config=cfg, logger=log)
    for label, p in paths.items():
        log.info("  %-10s -> %s" % (label, p))
'''

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", code],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env,
)
for line in proc.stdout:
    sys.stdout.write(line); sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 4 failed with exit code {proc.returncode}")

bulk_dir -> /content/drive/MyDrive/cs1090b_cl_bulk
  All 4 bulk CSVs already present in data/raw/cl_bulk - skipping
    courts-2026-03-31.csv.bz2  0.00 GB
    dockets-2026-03-31.csv.bz2  4.98 GB
    opinion-clusters-2026-03-31.csv.bz2  2.45 GB
    opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 5: Filter chain + extraction + manifest (Drive-persistent shards)
#
# Full pipeline: S3 discover → download → filter chain → extract → manifest → contract tests.
# Fast-path: if manifest.json + all shards are already present on Drive (e.g. pre-processed
# on Harvard ODD GPU Cluster L4 and copied to Drive), run_pipeline() returns immediately
# without re-running any stage. Otherwise it runs the full pipeline end-to-end.
import os, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Symlink shard_dir to Drive so shards persist across Colab sessions
DRIVE_SHARDS = pathlib.Path("/content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk")
DRIVE_SHARDS.mkdir(parents=True, exist_ok=True)
local_shards = pathlib.Path("data/raw/cl_federal_appellate_bulk")
local_shards.parent.mkdir(parents=True, exist_ok=True)
if local_shards.exists() and not local_shards.is_symlink():
    if local_shards.is_dir() and not any(local_shards.iterdir()):
        local_shards.rmdir()
    else:
        raise RuntimeError(f"{local_shards} exists and is not empty/symlink")
if not local_shards.exists():
    local_shards.symlink_to(DRIVE_SHARDS, target_is_directory=True)
print(f"shard_dir -> {local_shards.resolve()}")

# In-process pipeline — no subprocess, no embedded code string
import sys, logging
from src.config import PipelineConfig
from src.pipeline import run_pipeline, validate_pipeline
from src.exceptions import PipelineError
from src.timer import cell_timer


def _get_cell5_logger():
    lg = logging.getLogger("cell5")
    lg.setLevel(logging.INFO)
    if not lg.handlers:  # idempotent on cell re-run — fixes duplicate-handler bug
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg


logger = _get_cell5_logger()
cfg = PipelineConfig()

with cell_timer("Cell 5 - Pipeline (filter + extract + manifest)", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_pipeline (fast-path: skip if manifest + shards valid)")
    logger.info("=" * 60)
    manifest = run_pipeline(config=cfg, logger=logger)

    logger.info("\n" + "=" * 60)
    logger.info("  validate_pipeline (TDD contract tests)")
    logger.info("=" * 60)
    validate_pipeline(
        config=cfg, manifest_data=manifest,
        logger=logger, shard_strategy="sample",
    )  # raises PipelineError on failure — catchable by pytest.raises
    logger.info("OK contract tests passed")

    logger.info("\n" + "=" * 60)
    logger.info("  Summary")
    logger.info("=" * 60)
    logger.info("  snapshot: %s" % manifest["source_files"]["opinions"])
    logger.info("  git_rev:  %s" % manifest["run_metadata"]["git_revision"][:12])
    logger.info("  shards:   %d" % manifest["num_shards"])
    logger.info("  cases:    %s" % format(manifest["num_cases"], ","))
    logger.info("  scanned:  %s" % format(manifest.get("scanned", 0), ","))
    logger.info("  circuits: %d" % len(manifest.get("court_distribution", {})))
    tls = manifest.get("text_length_stats", {})
    if tls:
        logger.info("  text len: mean=%s median=%s p95=%s" % (
            format(tls.get("mean", 0), ","),
            format(tls.get("median", 0), ","),
            format(tls.get("p95", 0), ","),
        ))

In [ ]:
# Cell 6: Dataset readiness probe — full-corpus Polars scan, 8 gates
# Uses src.dataset_probe (v2.5.11, 303 contract tests, frozen ProbeConfig).
# Operates on the Drive-persistent shards from Cell 5. Fast-path: re-runs
# are cheap (~30s) because Polars scan_ndjson is memory-mapped.
import os, sys, logging, json, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from src.dataset_probe import run_probe, ProbeConfig
from src.timer import cell_timer

def _get_cell6_logger():
    lg = logging.getLogger("cell6")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg

logger = _get_cell6_logger()

shard_dir = pathlib.Path("data/raw/cl_federal_appellate_bulk")
out_path = pathlib.Path("logs/dataset_probe_report.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with cell_timer("Cell 6 - Dataset readiness probe", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_probe (full-corpus Polars scan, 8 gates)")
    logger.info("=" * 60)
    logger.info("  shard_dir: %s" % shard_dir.resolve())

    probe_cfg = ProbeConfig()
    report = run_probe(
        data_dir=str(shard_dir),
        subset=1_465_484,     # full corpus
        full_scan=True,       # always True in v2.5.11 (Polars mandatory)
        cfg=probe_cfg,
    )

    out_path.write_text(json.dumps(report.model_dump(), indent=2, default=str))
    logger.info("  report -> %s" % out_path)

    logger.info("\n" + "=" * 60)
    logger.info("  Probe summary")
    logger.info("=" * 60)
    summary = report.summary
    logger.info("  all_passed:     %s" % summary.get("all_passed"))
    logger.info("  passed_count:   %d" % summary.get("passed_count", 0))
    logger.info("  failed_block:   %d" % summary.get("failed_blocking_count", 0))
    logger.info("  subset_n:       %s" % format(report["subset_n"], ","))
    logger.info("  parse_errors:   %d" % report.get("parse_errors", 0))

    logger.info("\n  Gate results:")
    for gate_name, gate_result in report["gates"].items():
        status = "PASS" if gate_result.get("pass") else "FAIL"
        sev = gate_result.get("severity", "?")
        logger.info("    %-10s %s  (%s)" % (gate_name, status, sev))

    prov = report.get("provenance", {})
    logger.info("\n  probe_version: %s" % prov.get("probe_version"))
    logger.info("  polars_version: %s" % prov.get("polars_version"))
    logger.info("  full_scan: %s" % prov.get("full_scan"))

    if not summary.get("all_passed"):
        raise RuntimeError("probe gates failed — see report")
    logger.info("\nOK all gates passed — corpus cleared for Stage 3")

In [ ]:
# Cell 7: LePaRD dataset ingestion (Drive-persistent, self-healing)
# Uses scripts/ingest_lepard.py — pinned HF revision, atomic write, sidecar + manifest.
# Flow:
#   1. --verify-only fast-path on existing Drive artifact (Harvard ODD copy)
#   2. If verify fails, run default ingest which self-heals missing sidecar/manifest
#      from disk bytes (provenance_reconstructed=True); only downloads if JSONL itself
#      is absent or SHA256 doesn't match
#   3. Post-ingest --verify-only confirms final state
# Requires HF_TOKEN only if re-download is actually triggered.
import os, sys, subprocess, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Symlink data/raw/lepard to Drive
DRIVE_LEPARD = pathlib.Path("/content/drive/MyDrive/cs1090b_lepard")
DRIVE_LEPARD.mkdir(parents=True, exist_ok=True)

local_lepard = pathlib.Path("data/raw/lepard")
local_lepard.parent.mkdir(parents=True, exist_ok=True)
if local_lepard.exists() and not local_lepard.is_symlink():
    if local_lepard.is_dir() and not any(local_lepard.iterdir()):
        local_lepard.rmdir()
    else:
        raise RuntimeError(f"{local_lepard} exists and is not empty/symlink")
if not local_lepard.exists():
    local_lepard.symlink_to(DRIVE_LEPARD, target_is_directory=True)
print(f"lepard_dir -> {local_lepard.resolve()}")

# HF_TOKEN loaded lazily — only required if re-download triggers
def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return True
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab secrets")
        return True
    except Exception:
        return False

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    return proc.returncode

INGEST = [".venv/bin/python", "scripts/ingest_lepard.py"]

# Step 1: fast-path verify (no HF_TOKEN needed)
print("\n" + "=" * 60)
print("  Step 1: --verify-only (fast-path)")
print("=" * 60)
rc = _stream(INGEST + ["--verify-only"])

if rc == 0:
    print("\nOK artifact valid — skipping ingest (Harvard ODD copy in good state)")
else:
    # Step 2: self-heal OR re-ingest
    # Default invocation: idempotent, recomputes SHA256 from disk bytes,
    # reconstructs missing sidecar/manifest with provenance_reconstructed=True,
    # only downloads from HF if JSONL is absent or SHA256 mismatch.
    print("\n" + "=" * 60)
    print("  Step 2: self-heal / re-ingest")
    print("=" * 60)
    print("  ingest_lepard.py will:")
    print("    - recompute SHA256 from existing JSONL bytes")
    print("    - rebuild missing sidecar (.sha256) from computed digest")
    print("    - rebuild missing manifest.json with provenance_reconstructed=True")
    print("    - only download from HF if JSONL itself is missing or digest mismatch")

    if not _load_hf_token():
        print("  NOTE: HF_TOKEN not set — download path will fail if triggered")

    rc = _stream(INGEST)
    if rc != 0:
        raise SystemExit(f"LePaRD ingest failed with exit code {rc}")

    # Step 3: post-ingest verify
    print("\n" + "=" * 60)
    print("  Step 3: --verify-only (post-ingest confirmation)")
    print("=" * 60)
    rc = _stream(INGEST + ["--verify-only"])
    if rc != 0:
        raise SystemExit("post-ingest verify failed")

print("\nOK LePaRD artifact ready")
for p in sorted(DRIVE_LEPARD.glob("*")):
    size = p.stat().st_size
    print(f"  {p.name}  {size/1e9:.2f} GB" if size > 1e6 else f"  {p.name}  ({size} B)")

In [ ]:
# Cell 8: LePaRD ↔ CourtListener compatibility audit
# Uses src.lepard_cl_compat (56 TDD tests) — pure analysis, deterministic,
# CI-gate capable. Reports usable gold pair rate (both endpoints in CL corpus).
import os, sys, subprocess, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    return proc.returncode

print("=" * 60)
print("  src.lepard_cl_compat — pair-level compatibility audit")
print("=" * 60)

rc = _stream([".venv/bin/python", "-m", "src.lepard_cl_compat"])
if rc != 0:
    raise SystemExit(f"compat audit failed with exit code {rc}")

print("\nOK compatibility audit complete")

In [ ]:
# Cell 9: Data quality gate — NaN / encoding / parse audit on CL shards
# Uses scripts/audit_jsonl_nan.py — typed counters, W&B telemetry, CI gate.
# Operates on Drive-persistent shards from Cell 5. Produces CLEAN / REPAIRABLE /
# HARD_FAILURE / PARSE_FAILURE verdict. Fast: ~30s for 1.46M rows on 48 cores.
import os, sys, subprocess, json, pathlib
os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

shard_dir = pathlib.Path("data/raw/cl_federal_appellate_bulk")
if not shard_dir.exists() or not any(shard_dir.glob("shard_*.jsonl")):
    raise RuntimeError(f"no shards under {shard_dir} — run Cell 5 first")

def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, env=env,
    )
    lines = []
    for line in proc.stdout:
        sys.stdout.write(line); sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)

print("=" * 60)
print("  scripts/audit_jsonl_nan.py --json (data quality gate)")
print("=" * 60)

rc, output = _stream([
    ".venv/bin/python", "scripts/audit_jsonl_nan.py",
    "--input-dir", str(shard_dir),
    "--json",
])
if rc != 0:
    raise SystemExit(f"audit failed with exit code {rc}")

# Parse trailing JSON report
json_start = output.rfind("{")
if json_start >= 0:
    report = json.loads(output[json_start:])
    print("\n" + "=" * 60)
    print("  Audit summary")
    print("=" * 60)
    print(f"  verdict:       {report.get('gate_verdict')}")
    print(f"  clean_pct:     {report.get('clean_pct')}")
    print(f"  total_lines:   {report.get('total_lines'):,}")
    print(f"  total_shards:  {report.get('total_shards')}")
    print(f"  nan_lines:     {report.get('nan_lines')}")
    print(f"  nonfinite:     {report.get('nonfinite_lines')}")
    print(f"  decode_errors: {report.get('decode_error_lines')}")

    if report.get("gate_verdict") != "CLEAN":
        raise RuntimeError(f"audit gate failed: {report.get('gate_verdict')}")

print("\nOK all shards pass data quality gate")

## Pipeline Summary — Stages 1–3 + Readiness Gates

End-to-end CourtListener + LePaRD data pipeline for the Legal RAG project. Every cell is thin orchestration over TDD-covered modules in `src/` and `scripts/` — no business logic lives in the notebook.

### Cell map

| Cell | Stage | Purpose | Modules / Scripts |
|---|---|---|---|
| **1** | — | Title and scope | (markdown) |
| **2** | Bootstrap | Clone repo, install `uv`, create `.venv` (Python 3.11.9), sync pinned deps, write `.env` | `uv`, `pyproject.toml`, `uv.lock` |
| **3** | Preflight | Reproducibility config + GPU/dependency contract + preflight gate | `src.repro`, `src.environment`, `src.timer` |
| **4** | Stage 1 (acq) | Discover and download 4 CourtListener bulk CSVs from public S3 (~62 GB), persist to Google Drive, skip if present | `src.config`, `src.s3_discovery`, `src.bulk_download` |
| **5** | Stages 1–2 | Filter chain (courts → dockets → clusters), extract 1.46M opinions to 159 JSONL shards, write manifest with SHA-256 checksums, run TDD contract tests | `src.pipeline` (`run_pipeline`, `validate_pipeline`), which chains `src.manifest`, `src.s3_discovery`, `src.bulk_download`, `src.filter_chain`, `src.extract`, `src.validation`, `src.schemas`, `src.exceptions` |
| **6** | Stage 3 readiness | Full-corpus Polars `scan_ndjson` + 8 gates (schema, A7, A8, A9, A11, A12, A13, B6) | `src.dataset_probe` (v2.5.11, 303 contract tests) |
| **7** | Stage 1 (LePaRD) | Ingest LePaRD 4M training pairs from Hugging Face at pinned revision `0194f95c…`, with sidecar + manifest; self-heals missing sidecars from disk bytes | `scripts/ingest_lepard.py` (79 TDD tests) |
| **8** | Readiness | LePaRD ↔ CourtListener compatibility audit — reports usable gold pair rate (both endpoints present in CL corpus) | `src.lepard_cl_compat` (56 TDD tests) |
| **9** | Data quality gate | NaN / encoding / parse audit over all shards; verdict: `CLEAN / REPAIRABLE / HARD_FAILURE / PARSE_FAILURE` | `scripts/audit_jsonl_nan.py` |

### Persistence strategy

Google Colab ephemeral storage is replaced with Google Drive symlinks so no stage repeats across sessions:

```
data/raw/cl_bulk                   -> /content/drive/MyDrive/cs1090b_cl_bulk
data/raw/cl_federal_appellate_bulk -> /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
data/raw/lepard                    -> /content/drive/MyDrive/cs1090b_lepard
```

Fast-path behavior on reconnect:
- Cell 4: `download_file` skips any file already on Drive → ~0s
- Cell 5: `run_pipeline` reads manifest, validates shard SHA-256, returns immediately → ~1s
- Cell 6: Polars memory-mapped scan → ~30s regardless
- Cell 7: `--verify-only` checks sidecar + manifest + digest → ~5s
- Cell 9: 48-core parallel audit → ~30s

### Reproducibility guarantees

- **Python 3.11.9** pinned via `.python-version`
- **uv.lock** pins every dependency transitively (`torch 2.0.1+cu117`, `transformers 4.41.2`, `numpy 1.26.4`, …)
- **`src.repro.configure()`** sets `PYTHONHASHSEED=0`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, `torch.use_deterministic_algorithms(True)`, `cudnn.benchmark=False`, seeds Python/NumPy/torch CPU/CUDA RNGs to 0
- **LePaRD revision pinned** to `0194f95c3091acceab3b887c9b09ef432cf84052` (40-char SHA; mutable refs rejected)
- **Manifest checksums** — every shard's SHA-256 recorded; validation runs on every pipeline invocation
- **Contract tests** — `validate_pipeline` runs 13 `check_*` contract tests over sampled shards after extraction

### Downstream (not in this notebook)

README stages 4–7 — index generation (BM25 + FAISS), encoder training (BGE-M3 + Legal-BERT), sequential-loading evaluation (Tier A/B/C), W&B experiment tracking — are **not started** per the README and are correctly excluded from the data-pipeline notebook. They will be driven by `src.dataset_loader`, `src.lightning_datamodule`, `src.model_loader`, `src.split`, `src.wandb_logger`, and `src.hf_export` when that work begins.

### Verdict

> The full 1,465,484-record CourtListener Federal Appellate corpus plus the 4M-pair LePaRD training set are acquired, audited, probed, and verified compatible. The data pipeline is complete and reproducible across Colab sessions via Google Drive persistence. Ready for Stage 4 (indexing) whenever training begins.